# General Notebook

This notebook uses a custom kernel with all required dependencies installed.


In [1]:
# Check installed packages
import sys
print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")


Python version: 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:05:38) [MSC v.1929 64 bit (AMD64)]
Python executable: c:\Users\Hossein\.conda\envs\general_env\python.exe


In [5]:
# Import common libraries
try:
    import numpy as np
    import boto3
    import requests
    from tqdm import tqdm
    import zarr
    print("All dependencies imported successfully!")
except ImportError as e:
    print(f"Missing dependency: {e}")


All dependencies imported successfully!


In [6]:
# Read biomedvis-6gb data and print metadata
import zarr
from pathlib import Path
data_path = Path("BioProject/Data/biomedvis-6gb/0")
print(f"{'='*60}\nDataset: biomedvis-6gb\n{'='*60}")
try:
    zarr_group = zarr.open(str(data_path), mode='r')
    print(f"\nPath: {data_path}")
    print(f"Type: Zarr Group")
    print(f"Arrays: {list(zarr_group.keys())}")
    # Print array metadata
    for key in zarr_group.keys():
        arr = zarr_group[key]
        print(f"\n  [{key}] Shape: {arr.shape} | Dtype: {arr.dtype} | Chunks: {arr.chunks}")
    # Print key metadata from attributes
    if hasattr(zarr_group, 'attrs') and zarr_group.attrs:
        attrs = zarr_group.attrs
        if 'multiscales' in attrs:
            scales = attrs['multiscales'][0]
            print(f"\n  Name: {scales.get('name', 'N/A')}")
            print(f"  Axes: {[ax['name'] for ax in scales.get('axes', [])]}")
            print(f"  Resolution levels: {len(scales.get('datasets', []))}")
        if 'omero' in attrs:
            channels = attrs['omero'].get('channels', [])
            channel_names = [c['label'] for c in channels]
            print(f"  All channels ({len(channel_names)}): {channel_names}")
except Exception as e:
    print(f"Error reading zarr data: {e}")

Dataset: biomedvis-6gb
Error reading zarr data: nothing found at path ''


In [16]:
# Load data as dask array
import dask.array as da
import zarr
from pathlib import Path
from IPython.display import HTML, display

# Path relative to notebook location (BioProject folder)
path = Path("Data/biomedvis-6gb/0")

# Check if path exists
if not path.exists():
    # Try alternative path
    path = Path("BioProject/Data/biomedvis-6gb/0")

# Open zarr group directly
zarr_group = zarr.open(str(path), mode='r')
store = zarr_group.store
daskArray = da.from_zarr(store, component="3")  # Using component "3" from biomedvis-6gb

# Display detailed information in Colab-like format
def format_bytes(bytes_size):
    """Format bytes to human readable format"""
    for unit in ['B', 'KiB', 'MiB', 'GiB', 'TiB']:
        if bytes_size < 1024.0:
            return f"{bytes_size:.2f} {unit}"
        bytes_size /= 1024.0
    return f"{bytes_size:.2f} PiB"

# Calculate number of chunks
num_chunks = daskArray.numblocks
total_chunks = 1
for n in num_chunks:
    total_chunks *= n

# Calculate chunk size in bytes
chunk_size_bytes = 1
for cs in daskArray.chunksize:
    chunk_size_bytes *= cs
chunk_size_bytes *= daskArray.dtype.itemsize

# Display information in Colab-like format
print("Array\tChunk")
print(f"Bytes\t{format_bytes(daskArray.nbytes)}\t{format_bytes(chunk_size_bytes)}")
print(f"Shape\t{daskArray.shape}\t{daskArray.chunksize}")
print(f"Dask graph\t{total_chunks} chunks in 2 graph layers")
print(f"Data type\t{daskArray.dtype} numpy.ndarray")
print()

Array	Chunk
Bytes	23.72 GiB	1.34 MiB
Shape	(1, 70, 194, 688, 1363)	(1, 1, 1, 688, 1024)
Dask graph	27160 chunks in 2 graph layers
Data type	>u2 numpy.ndarray



In [15]:
# Reading the Physical Size of the Sample
from pathlib import Path
import zarr

# Path relative to notebook location
path = Path("Data/biomedvis-6gb/0")


# Read zarr group and extract physical size from multiscales metadata
zarr_group = zarr.open(str(path), mode='r')
attrs = zarr_group.attrs

if 'multiscales' in attrs:
    scales = attrs['multiscales'][0]
    datasets = scales.get('datasets', [])
    if datasets:
        transforms = datasets[0].get('coordinateTransformations', [])
        if transforms:
            scale = transforms[0].get('scale', [])  # [t, c, z, y, x] in micrometers
            if len(scale) >= 5:
                physical_size_x = scale[4]  # x dimension
                physical_size_y = scale[3]  # y dimension  
                physical_size_z = scale[2]  # z dimension
                print(f"Physical Size X: {physical_size_x} μm")
                print(f"Physical Size Y: {physical_size_y} μm")
                print(f"Physical Size Z: {physical_size_z} μm")
                print(f"\nPhysical Size: ({physical_size_x}, {physical_size_y}, {physical_size_z}) μm")
                # Store for later use
                ome_xml_physical_size = (physical_size_x, physical_size_y, physical_size_z)


Physical Size X: 0.13999999999999999 μm
Physical Size Y: 0.14 μm
Physical Size Z: 0.28 μm

Physical Size: (0.13999999999999999, 0.14, 0.28) μm


In [17]:
# Getting the raw data from Channel CD31
# CD31 is at channel index 19 (0-indexed)
# daskArray shape: (t, c, z, y, x) = (1, 70, 194, 688, 1363)

# Get data for t=0, channel=19 (CD31), all z, y, x
# Uses daskArray from previous cell (Cell 4)
data = daskArray[0, 19, :, :, :].compute()

print(f"Channel: CD31 (index 19)")
print(f"Data shape: {data.shape}")
print(f"Data dtype: {data.dtype}")
print(f"Data range: [{data.min()}, {data.max()}]")
print(f"Data mean: {data.mean():.2f}")
print(f"Data size: {data.nbytes / (1024**2):.2f} MB")

data.shape


Channel: CD31 (index 19)
Data shape: (194, 688, 1363)
Data dtype: >u2
Data range: [0, 32737]
Data mean: 13.17
Data size: 346.99 MB


(194, 688, 1363)

## 3D Visualization Options

**Best Methods for Large Volume Data:**

1. **Napari** (Recommended) - Interactive volume visualization, supports large datasets with lazy loading
2. **Plotly** - WebGL-based 3D visualization, good for interactive notebooks
3. **Three.js/WebGL** - Custom HTML viewer with volume rendering
4. **PyVista** - 3D scientific visualization

**Note:** Due to large size (194 x 688 x 1363), we'll use downsampling or volume rendering techniques.


In [18]:
# Method 1: Napari - Best for interactive volume visualization
# Install: pip install napari[all]

try:
    import napari
    from skimage import exposure
    
    # Select channel (channel index 0 = first channel)
    channel_idx = 0
    print(f"Loading channel {channel_idx} for 3D visualization...")

    
    
    # Load channel data (downsample if needed for better performance)
    # Option 1: Use full resolution (may be slow)
    # channel_data = daskArray[0, channel_idx, :, :, :].compute()
    
    # Option 2: Downsample for better performance (recommended)
    # Downsample by factor of 2 in each dimension
    channel_data = daskArray[0, channel_idx, ::1, ::1, ::1].compute()
    print(f"Data shape (downsampled): {channel_data.shape}")
    print(f"Data range: [{channel_data.min()}, {channel_data.max()}]")
    
    # Normalize data for visualization (0-255 range)
    channel_data_normalized = exposure.rescale_intensity(
        channel_data, 
        in_range=(channel_data.min(), channel_data.max()),
        out_range=(0, 255)
    ).astype('uint8')
    
    # Create napari viewer
    viewer = napari.Viewer()
    viewer.add_image(
        channel_data_normalized,
        name=f'Channel {channel_idx}',
        rendering='iso',  # Iso-surface rendering
        opacity=0.7,
        contrast_limits=[0, 255]
    )
    print("\nNapari viewer opened. Use mouse to rotate/zoom the 3D volume.")
    print("Controls:")
    print("  - Left mouse: Rotate")
    print("  - Right mouse: Pan")
    print("  - Scroll: Zoom")
    print("  - 'Iso' button: Toggle iso-surface rendering")
    
except ImportError:
    print("Napari not installed. Install with: pip install napari[all]")
    print("Or try Method 2 (Plotly) below.")
except Exception as e:
    print(f"Error with Napari: {e}")
    print("Trying alternative method...")


Loading channel 0 for 3D visualization...
Data shape (downsampled): (194, 688, 1363)
Data range: [0, 33997]

Napari viewer opened. Use mouse to rotate/zoom the 3D volume.
Controls:
  - Left mouse: Rotate
  - Right mouse: Pan
  - Scroll: Zoom
  - 'Iso' button: Toggle iso-surface rendering


In [29]:
# Method 2: Plotly - WebGL 3D Visualization
import plotly.graph_objects as go
import numpy as np
from plotly.offline import plot
from pathlib import Path
from IPython.display import HTML, display

channel_idx = 0

# Load and downsample data
channel_data = daskArray[0, channel_idx, ::2, ::4, ::4].compute()
print(f"Channel {channel_idx} shape: {channel_data.shape}")

# Normalize to 0-255
channel_data_norm = ((channel_data - channel_data.min()) / 
                     (channel_data.max() - channel_data.min()) * 255).astype(np.uint8)

# Create point cloud (sample every 2nd point)
sample_step = 2
z_coords, y_coords, x_coords = np.meshgrid(
    np.arange(channel_data_norm.shape[0])[::sample_step],
    np.arange(channel_data_norm.shape[1])[::sample_step],
    np.arange(channel_data_norm.shape[2])[::sample_step],
    indexing='ij'
)
sampled_data = channel_data_norm[::sample_step, ::sample_step, ::sample_step]

# Filter points with intensity > 10th percentile
values = sampled_data.flatten()
threshold = np.percentile(values, 10)
mask = values > threshold

# Create visualization
fig = go.Figure(data=go.Scatter3d(
    x=x_coords.flatten()[mask],
    y=y_coords.flatten()[mask],
    z=z_coords.flatten()[mask],
    mode='markers',
    marker=dict(
        size=2,
        color=values[mask],
        colorscale='Viridis',
        opacity=0.6,
        showscale=True
    )
))

fig.update_layout(
    title=f'3D Visualization - Channel {channel_idx}',
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
    width=1200,
    height=900
)

# Save and display
output_dir = Path("visualization_data")
output_dir.mkdir(exist_ok=True)
html_file = output_dir / f"plotly_3d_channel_{channel_idx}.html"
plot(fig, filename=str(html_file), auto_open=False)

print(f"✓ Saved to: {html_file}")
try:
    display(HTML(plot(fig, output_type='div', include_plotlyjs='cdn')))
except:
    pass


Channel 0 shape: (97, 172, 341)
✓ Saved to: visualization_data\plotly_3d_channel_0.html


In [31]:
# Method 3: Prepare data for Three.js WebGL viewer (export to format usable by web)
# This creates a compressed representation that can be loaded in a Three.js viewer

import json
import numpy as np
from pathlib import Path

def prepare_data_for_threejs(channel_idx=0, downsample_factor=4):
    """
    Prepare channel data for Three.js volume rendering.
    Returns data in a format suitable for web transfer.
    """
    print(f"Preparing channel {channel_idx} data for Three.js...")
    
    # Load and downsample
    channel_data = daskArray[0, channel_idx, ::downsample_factor, ::downsample_factor, ::downsample_factor].compute()
    print(f"Shape: {channel_data.shape}")
    
    # Normalize to 0-255
    data_min, data_max = channel_data.min(), channel_data.max()
    channel_data_norm = ((channel_data - data_min) / (data_max - data_min) * 255).astype(np.uint8)
    
    # Save metadata
    metadata = {
        'shape': channel_data.shape,
        'dataRange': [int(data_min), int(data_max)],
        'downsampleFactor': downsample_factor,
        'channel': channel_idx
    }
    
    # Option 1: Save as compressed numpy array (can be loaded in JavaScript with pako.js)
    output_dir = Path("visualization_data")
    output_dir.mkdir(exist_ok=True)
    
    # Save metadata
    with open(output_dir / f"channel_{channel_idx}_metadata.json", 'w') as f:
        json.dump(metadata, f, indent=2)
    
    # Save data as binary (can be loaded with fetch in JS)
    channel_data_norm.tofile(output_dir / f"channel_{channel_idx}_data.raw")
    
    print(f"\nData prepared for Three.js viewer!")
    print(f"Files saved to: visualization_data")
    print(f"  - channel_{channel_idx}_metadata.json")
    print(f"  - channel_{channel_idx}_data.raw")
    print(f"\nYou can now create an HTML file with Three.js to load and render this data.")
    print(f"Data size: {channel_data_norm.nbytes / (1024**2):.2f} MB")
    
    return metadata, channel_data_norm

# Prepare data for channel 0
metadata, prepared_data = prepare_data_for_threejs(channel_idx=0, downsample_factor=4)


Preparing channel 0 data for Three.js...
Shape: (49, 172, 341)

Data prepared for Three.js viewer!
Files saved to: visualization_data
  - channel_0_metadata.json
  - channel_0_data.raw

You can now create an HTML file with Three.js to load and render this data.
Data size: 2.74 MB
